# 标签经济含义核查
固定原阈值与持仓，复算主标签并对照现金/杠杆端点。只检查标签是否同义，不评估预测能力、不重新拟合、不修改现役标签。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/label_semantics_20261014';plan=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013';qlib.init(provider_uri=str(provider),region='us')
s,audit=_source(root,plan['source'],['inputs/daily_frame']);frame=s['inputs/daily_frame'];rx=frame.soxx_forward_open_return;rl=frame.soxl_forward_open_return
xx=pd.concat([rx.shift(-i) for i in range(5)],axis=1);xl=pd.concat([rl.shift(-i) for i in range(5)],axis=1);valid=xx.notna().all(axis=1)&xl.notna().all(axis=1)
xx.columns=range(5);xl.columns=range(5)
benchmark=np.log1p(xx.clip(lower=-.99)).sum(axis=1,min_count=5);leveraged=np.log1p(xl.clip(lower=-.99)).sum(axis=1,min_count=5)
mixed=np.log1p((.27*xl+.63*xx).clip(lower=-.99)).sum(axis=1,min_count=5)
alpha=mixed-benchmark
np.testing.assert_allclose(alpha.loc[valid],frame.relative_alpha_5d.loc[valid],rtol=1e-7,atol=1e-7)
minimum=((1+xx).cumprod(axis=1)-1).min(axis=1);tail=minimum.le(-.06).astype(float).where(valid)
np.testing.assert_allclose(tail.loc[valid],frame.tail_event_5d.loc[valid],rtol=0,atol=0)
labels=pd.DataFrame({'label_available_date':pd.to_datetime(frame.exit_date).shift(-4),'primary_tail':tail,'primary_mixed_alpha_positive':alpha.gt(0).astype(float).where(valid),'cash_beats_soxx':benchmark.lt(0).astype(float).where(valid),'soxl_beats_soxx':(leveraged-benchmark).gt(0).astype(float).where(valid),'primary_mixed_alpha':alpha,'direct_leverage_alpha':leveraged-benchmark,'direct_cash_advantage':-benchmark})
assert labels.loc[valid,'label_available_date'].notna().all()
rows=[];confusions=[]
for name,start,end in [('mature_training','2011-01-03','2022-12-30'),('reused_later','2023-01-03',plan['as_of'])]:
 mask=valid&frame.index.to_series().ge(start)&labels.label_available_date.le(pd.Timestamp(end));part=labels.loc[mask]
 for left,right in [('primary_tail','cash_beats_soxx'),('primary_mixed_alpha_positive','soxl_beats_soxx')]:
  row={'window':name,'left_label':left,'right_label':right,'observations':len(part),'left_positive_rate':float(part[left].mean()),'right_positive_rate':float(part[right].mean()),'disagreement_rate':float(part[left].ne(part[right]).mean())};rows.append(row)
  for l in [0,1]:
   for r in [0,1]:confusions.append({'window':name,'left_label':left,'right_label':right,'left_value':l,'right_value':r,'rows':int((part[left].eq(l)&part[right].eq(r)).sum())})
contracts=[{'output':'primary_tail_probability','asset':'SOXX unlevered','clock':'five next-open-to-following-open daily returns','event':'minimum entry-relative cumulative return <= -6%','not_equivalent_to':'cash outperforms SOXX at terminal date; peak-to-trough drawdown'},
 {'output':'primary_alpha_probability','asset':'27% SOXL + 63% SOXX + 10% cash','clock':'five daily open-to-open returns, fixed weights each interval','event':'mixed portfolio compounded log wealth > SOXX','not_equivalent_to':'SOXL outperforms SOXX; deployable current 15%-cash-floor portfolio'},
 {'output':'auxiliary_tail_probability','asset':'synthetic daily-reset 2x single constituent','clock':'next five close-to-close returns','event':'entry-relative minimum <= -12% OR terminal <= -15%; 95bp annual fee assumption','not_equivalent_to':'actual SOXL/SOXX cash-timing probability'},
 {'output':'auxiliary_upside_probability','asset':'synthetic daily-reset 2x single constituent versus same constituent 1x','clock':'next five close-to-close returns','event':'2x terminal return > 1x terminal return','not_equivalent_to':'primary mixed-portfolio alpha probability'},
 {'output':'cash_weight','asset':'SOXL/SOXX/cash allocation','clock':'model output before smoothing/controllers','event':'portfolio weight learned from joint objective, not a probability target','not_equivalent_to':'probability that cash beats SOXX'}]
summary={'experiment':card.name,'decision':'labels_have_distinct_economic_targets_no_automatic_relabeling','new_fits':0,'new_backtests':0,'parameter_searches':0,'objective_achieved':False,'primary_label_reconstruction_verified':True,'label_disagreements_are_not_forecast_performance':True,'provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/recorder.html','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily_labels','sources/*']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='no_fit_label_contract',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0)
 rec.save_objects(artifact_path='inputs',plan=plan,source_audit=audit)
 rec.save_objects(artifact_path='analysis',daily_labels=labels,comparison=pd.DataFrame(rows),confusions=pd.DataFrame(confusions),contracts=contracts,summary=summary)
 rec.save_objects(artifact_path='sources',primary_labels=(root/'src/workflow/cash_floor_relative_feature_reintegration.py').read_text(),tail_labels=(root/'src/workflow/direct_continuous_soxx_allocation.py').read_text(),auxiliary_labels=(root/'src/factor/soxx_pit_tail_factors.py').read_text())
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');atomic_json(contracts,card/'label_contract.json');pd.DataFrame(rows).to_csv(card/'label_comparison.csv',index=False);pd.DataFrame(confusions).to_csv(card/'label_confusion_counts.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False))
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256'];print(pd.DataFrame(rows).to_string(index=False))


# 标签不同义：尾部事件不能解释成现金优势概率

0拟合、0回测、0阈值搜索。按原有定义重建主标签并与冻结Recorder值核对；未修改训练标签、模型权重或任何生产输出。标签之间的列联统计不是预测成绩，也不以哪种定义历史收益更好选择标签。

主尾部标签是SOXX未来五段日线开盘到开盘路径中，相对入场累计收益最低值不高于-6%；不是峰谷最大回撤，也不是五日终点现金胜率。成熟训练3014行中尾部阳性6.77%，终点现金胜SOXX为42.47%，两者不同36.03%；后续复用940行分别11.17%、40.32%，不同30.85%。轻度终点下跌可能没有尾部事件，路径下跌后也可能恢复，不能直接把tail_probability当作现金优势概率。

主alpha标签来自每日固定27%SOXL、63%SOXX、10%现金参考组合的五段复利收益相对SOXX。不是纯SOXL相对SOXX，也不是当前至少15%现金约束下的实际配置。与纯SOXL相对SOXX正负标签，训练/复用不同仅1.19%/0.96%，因此不据字段命名差异夸大其对失败的解释力。这个参考组合是旧标签合同，不自动成为可执行目标。

辅助尾部是历史成分股日重置合成2x、五个未来收盘到收盘区间、95bp年费用假设下的路径最低收益<=-12%或终点<=-15%；辅助上涨是合成2x终点超过该成分股1x。它与主标的的资产、杠杆、时钟和事件阈值不同，只能作为迁移任务，不能把输出理解为实际SOXL或SOXX的相同概率。

现金权重则是联合配置损失产生的持仓比例，不是直接监督出来的概率。前一轮局部梯度接近正交也不能替代这些语义边界；本卡不证明辅助迁移有害，更不支持直接删除辅助任务或强制反转分数。

两组统计均按原标签最终exit_date成熟，不把未知末端变成0。采用原日线开盘标签，不能与09:47执行点的另一批标签不加说明地混比。现金比较为零利息、未扣换仓费的事件定义，非IBKR净收益策略。

决定：labels_have_distinct_economic_targets_no_automatic_relabeling。明确了五个输出/任务的资产、时钟、事件与禁止解释，见label_contract.json。稳定净超额未达到，原冻结模型不变。后续先追踪这些输出是否真的被下游当成错误的现金择时分数使用；如果没有，不把标签不同义本身当作已找到亏损原因，也不重复已有现金H1/H5弱排序实验。

[标签合同](label_contract.json) · [定义差异](label_comparison.csv) · [完整列联计数](label_confusion_counts.csv)

Recorder `785721be80234b0ebc45b77a4ed703bc`。
